In [1]:
#Task 2
# Geçmişin ortalamasını üç yoldan al: for döngüsüyle, lower triangular
# matrisle (torch.tril) çarparak, softmax'la. Üçünün aynı sonucu verdiğini
# torch.allclose ile göster. Matris çarpımının neden ağırlıklı ortalama
# olduğunu kendi cümlelerinle yaz. (42:13 - 58:26)

In [9]:
import torch 
from torch.nn import functional as F
torch.manual_seed(1337)
B, T, C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [10]:
#filling with for loop the most primitive way.
xbow = torch.zeros(B,T,C)
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev,0)

In [11]:
xbow[0],x[0]

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]]))

In [12]:
#lower triangular matrix vs for loop
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (T,T) @ (B,T,C) -->(B,T,C)
torch.allclose(xbow,xbow2)

True

In [13]:
#softmax
tril = torch.tril(torch.ones(T,T))
wei2 = torch.zeros(T,T)
wei2 = wei2.masked_fill(tril == 0, float('-inf'))
wei2 = F.softmax(wei2,dim=-1)
xbow3 = wei2 @ x
torch.allclose(xbow,xbow3)

True

In [14]:
#another softmax with triu
triu = torch.triu(torch.ones(T,T),diagonal=1) 
wei3 = torch.zeros(T,T)
wei3 = wei3.masked_fill(triu == 1, float('-inf'))
wei3 = F.softmax(wei3,dim=-1)
xbow4 = wei3 @ x 
torch.allclose(xbow,xbow4)

True

In [17]:
#again tril but fails. softmax just tried to inf - inf and got nan
tril2 = torch.tril(torch.ones(T,T))
wei4 = torch.zeros(T,T)
wei4 = wei4.masked_fill(tril2 ==1, float('inf'))
wei4_maxed = F.softmax(wei4,dim=-1)     
wei4_maxed, wei4

(tensor([[nan, nan, nan, nan, nan, nan, nan, nan],
         [nan, nan, nan, nan, nan, nan, nan, nan],
         [nan, nan, nan, nan, nan, nan, nan, nan],
         [nan, nan, nan, nan, nan, nan, nan, nan],
         [nan, nan, nan, nan, nan, nan, nan, nan],
         [nan, nan, nan, nan, nan, nan, nan, nan],
         [nan, nan, nan, nan, nan, nan, nan, nan],
         [nan, nan, nan, nan, nan, nan, nan, nan]]),
 tensor([[inf, 0., 0., 0., 0., 0., 0., 0.],
         [inf, inf, 0., 0., 0., 0., 0., 0.],
         [inf, inf, inf, 0., 0., 0., 0., 0.],
         [inf, inf, inf, inf, 0., 0., 0., 0.],
         [inf, inf, inf, inf, inf, 0., 0., 0.],
         [inf, inf, inf, inf, inf, inf, 0., 0.],
         [inf, inf, inf, inf, inf, inf, inf, 0.],
         [inf, inf, inf, inf, inf, inf, inf, inf]]))

In [27]:
#matrix mul acts like weighted sum because before mul we are manipulating the matrix inputs. 
#and also matrix multiplication can explain as linear combination of columns it acts like weighted sum
#[1       0     0]     [a]          [1]          [0]         [0]          [  1* a  ]    
#[0.5   0.5     0] @   [b]  -> a *  [.5] + b *  [.5]  + c *  [0]     =    [0.5(a+b)   ]  0.5*2 =1
#[0.33 0.33 0.33]      [c]         [.33]        [.33]        [0.33]       [0.33(a+b+c)]  0.33*3 =1

#In the softmax version, since softmax is a normalization operator, we fill the bottom of the diagonal with same number and the top (excluding the diagonal) with -inf. 
#-inf makes e^-inf = the top diagonal zero, and since other non -infinite values before softmax are equal, the probability for this row is distributed equally. thats why its also acts like weighted sum 